In [12]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")
library("parallel")
dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }

dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }

num_cores <- 24
register(MulticoreParam(num_cores))
bpparam <- MulticoreParam(num_cores, log = TRUE, stop.on.error = FALSE)


Load experiment

# All Brain

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_barcodes.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)
write.csv(test_results,"20241126_MAD_Brain.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 2
Node: 23
Timestamp: 2024-11-26 23:51:17.814263
Success: TRUE

Task duration:
   user  system elapsed 
  4.477   0.322   5.195 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6182068 330.2   11623467 620.8 10290760 549.6
Vcells 11239709  85.8   19179330 146.4 19178645 146.4

Log messages:
INFO [2024-11-26 23:51:12] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 21
Node: 4
Timestamp: 2024-11-26 23:51:18.666053
Success: TRUE

Task duration:
   user  system elapsed 
  4.356   0.394   5.327 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6196156 331.0   11623467 620.8 10290760 549.6
Vcells 11317766  86.4   19179330 146.4 19178645 146.4

Log messages:
INFO [2024-11-26 23:51:13] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-11-26 23:5

# Brain Region With Baseline

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_barcodes.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  ) 
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele, 
                            rnaDesign = ~ Animal+Tissue
                            )
alpha <- getAlpha(obj, by.factor = "Tissue")
for (col_names in names(alpha)){   
    result <- testEmpirical(obj = obj,statistic = alpha[,col_names])
    row.names(result)<-row.names(alpha)
    write.csv(result, paste("enhancer_activities/MAD_OneTail_NoControl/20241126_MAD_",col_names,".csv",sep=""))}

Fitting model...

############### LOG OUTPUT ###############
Task: 6
Node: 19
Timestamp: 2024-11-26 23:56:59.03547
Success: TRUE

Task duration:
   user  system elapsed 
 11.559   0.431  12.170 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6182779 330.2   11623467 620.8 10290760 549.6
Vcells 11243610  85.8   23095196 176.3 23094931 176.3

Log messages:
INFO [2024-11-26 23:56:46] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 13
Node: 12
Timestamp: 2024-11-26 23:57:00.036205
Success: TRUE

Task duration:
   user  system elapsed 
 11.514   0.494  12.591 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6193497 330.8   11623467 620.8 10290760 549.6
Vcells 11309302  86.3   23095196 176.3 23094931 176.3

Log messages:
INFO [2024-11-26 23:56:47] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 14
Node: 11
Timestamp: 2024-11-26 23:

# THP1 Macrophage All

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/THP1Macrophage_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/THP1Macrophage_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]
##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
###############################################################
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)
write.csv(test_results,"20241126_MAD_THP1Macrophage.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-11-26 23:53:58.833601
Success: TRUE

Task duration:
   user  system elapsed 
128.798   1.222 136.082 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6182684 330.2   11623467 620.8 10290760 549.6
Vcells 12210360  93.2   23095196 176.3 19179330 146.4

Log messages:
INFO [2024-11-26 23:51:42] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 19
Node: 6
Timestamp: 2024-11-26 23:54:09.33334
Success: TRUE

Task duration:
   user  system elapsed 
140.227   0.780 146.762 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6183369 330.3   11623467 620.8 10290760 549.6
Vcells 12218802  93.3   23095196 176.3 19179330 146.4

Log messages:
INFO [2024-11-26 23:51:42] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 12
Node: 13
Timestamp: 2024-11-26 23:5

# THP1 Macrophage with Baseline

In [ ]:

#############################################################################################################################################

df_DNA <- read.csv("read_counts_R1R2/THP1Macrophage_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/THP1Macrophage_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodes.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]
##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
###############################################################
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ Tissue
                            )

alpha <- getAlpha(obj, by.factor = "Tissue")
for (col_names in names(alpha)){   
    result <- testEmpirical(obj = obj,statistic = alpha[,col_names])
    row.names(result)<-row.names(alpha)
    write.csv(result, paste("enhancer_activities/MAD_OneTail_NoControl/20240824_MAD_",col_names,".csv",sep=""))}

In [ ]:
#############################################################################################################################################
df_DNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_DNA_matched_barcodes_reshaped_altref_20250121.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/THP1_LPSIFNGvsNaive_RNA_matched_barcodes_reshaped_altref_20250121.csv", header=TRUE,row.names=1)
df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1_LPSIFNGvsNaive_barcodes_20250121.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]


##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

# HMC3 All

In [ ]:

df_DNA <- read.csv("read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_barcodes_interaction.csv", header=TRUE,row.names=1)
df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]
df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)


#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]
##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
###############################################################
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)
write.csv(test_results,"20241126_MAD_HMC3.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-11-26 23:55:44.645387
Success: TRUE

Task duration:
   user  system elapsed 
 77.894   0.713  83.006 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6182741 330.2   11623467 620.8 10290760 549.6
Vcells 11954782  91.3   23095196 176.3 23094931 176.3

Log messages:
INFO [2024-11-26 23:54:21] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-11-26 23:55:50.66767
Success: TRUE

Task duration:
   user  system elapsed 
 84.376   0.936  89.620 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6183426 330.3   11623467 620.8 10290760 549.6
Vcells 11962891  91.3   23095196 176.3 23094931 176.3

Log messages:
INFO [2024-11-26 23:54:21] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 9
Node: 16
Timestamp: 2024-11-26 23:55

# HMC3 With Baseline

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/HMC3_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HMC3_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HMC3_barcodes_interaction.csv", header=TRUE,row.names=1)
df_DNA <- df_DNA[, !grepl("ZC23", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC23", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC23", rownames(annot_DNA)), ]
df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)


#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]
##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]
###############################################################
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ Tissue
                            )
alpha <- getAlpha(obj, by.factor = "Tissue")
for (col_names in names(alpha)){   
    result <- testEmpirical(obj = obj,statistic = alpha[,col_names])
    row.names(result)<-row.names(alpha)
    write.csv(result, paste("enhancer_activities/MAD_OneTail_NoControl/20240823_MAD_",col_names,".csv",sep=""))}


Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-13 00:01:35.068063
Success: TRUE

Task duration:
   user  system elapsed 
122.448   0.707 130.084 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6170440 329.6   11607041 619.9  8190798 437.5
Vcells 11920832  91.0   23110480 176.4 19192055 146.5

Log messages:
INFO [2024-08-12 23:59:25] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-08-13 00:01:45.24344
Success: TRUE

Task duration:
   user  system elapsed 
132.251   0.726 140.828 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6171123 329.6   11607041 619.9  8190798 437.5
Vcells 11929336  91.1   23110480 176.4 19192055 146.5

Log messages:
INFO [2024-08-12 23:59:24] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp: 2024-08-13 00:01

# THP1 Monocytes

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/THP1mono20240412_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/THP1mono20240412_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Monocyte20240412_barcodes.csv", header=TRUE,row.names=1)
df_DNA <- df_DNA[, !grepl("ZC37|ZC60", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC37|ZC60", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC37|ZC60", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)
write.csv(test_results,"20240729_MAD_THP1Monocyte.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 23
Node: 1
Timestamp: 2024-07-29 02:12:54
Success: TRUE

Task duration:
   user  system elapsed 
  7.907   0.936   9.765 

Memory used:
          used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells 5388129 287.8    8040154 429.4  7409332 395.8
Vcells 9436176  72.0   18729336 142.9 17802400 135.9

Log messages:
INFO [2024-07-29 02:12:44] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 4
Node: 20
Timestamp: 2024-07-29 02:12:55
Success: TRUE

Task duration:
   user  system elapsed 
  8.909   0.797  10.456 

Memory used:
          used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells 5403562 288.6    8040154 429.4  7409332 395.8
Vcells 9521909  72.7   18729336 142.9 17802400 135.9

Log messages:
INFO [2024-07-29 02:12:44] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 19
Timestamp: 2024-07-29 02:12:55
Success: TRUE



# HEK293T

In [ ]:

df_DNA <- read.csv("read_counts_R1R2/HEK293T_DNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/HEK293T_RNA_matched_barcodes_reshaped_altref.csv", header=TRUE,row.names=1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_HEK293T_barcodes.csv", header=TRUE,row.names=1)
df_DNA <- df_DNA[, !grepl("ZC65|ZC66", colnames(df_DNA))]
df_RNA <- df_RNA[, !grepl("ZC65|ZC66", colnames(df_RNA))]
annot_DNA <- annot_DNA[!grepl("ZC65|ZC66", rownames(annot_DNA)), ]

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]
##############################################################
total_columns <- ncol(df_DNA)
selected_columns <- seq(from = 1, to = total_columns)[((seq(from = 1, to = total_columns) - 1) %% 32) < 15]
df_DNA <- df_DNA[, selected_columns]
df_RNA <- df_RNA[, selected_columns]
annot_DNA <- annot_DNA[selected_columns,]

###############################################################
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )

#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)

write.csv(test_results,"20240716_MAD_HEK293T.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 23
Node: 1
Timestamp: 2024-07-16 13:44:31
Success: TRUE

Task duration:
   user  system elapsed 
 21.435   0.379  23.378 

Memory used:
          used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells 5390477 287.9    8459960 451.9  8439507 450.8
Vcells 9519791  72.7   18606928 142.0 18606643 142.0

Log messages:
INFO [2024-07-16 13:44:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 19
Timestamp: 2024-07-16 13:44:32
Success: TRUE

Task duration:
   user  system elapsed 
 23.312   0.258  24.704 

Memory used:
          used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells 5391585 288.0    8459960 451.9  8439507 450.8
Vcells 9528094  72.7   18606928 142.0 18606643 142.0

Log messages:
INFO [2024-07-16 13:44:07] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 17
Node: 7
Timestamp: 2024-07-16 13:44:33
Success: TRUE



# Neuron


In [10]:
df_DNA <- read.csv("/Volumes/T7/ad_mpra_chen/outputs/read_counts_R1R2/NeuronPseudobarcodes_DNA_matched_barcodes_reshaped_altref_controls.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("/Volumes/T7/ad_mpra_chen/outputs/read_counts_R1R2/NeuronPseudobarcodes_RNA_matched_barcodes_reshaped_altref_controls.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)
annot_DNA <-read.csv("/Volumes/T7/ad_mpra_chen/annotation_barcodes/mpra3_annot_NeuronPseudobarcodes_altref_barcodes.csv", header=TRUE,row.names=1)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

annot_DNA <- annot_DNA [!grepl("REF", rownames(annot_DNA )), ]

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeQuantification(
                            obj = obj, 
                            dnaDesign = ~ Barcode_Allele+Test, 
                            rnaDesign = ~ 1
                            )
test_results <- testEmpirical(obj = obj,twoSided = FALSE)
write.csv(test_results,"20260625_MAD_Neuron_Pseudobarcode_control_R1R2.csv")


Fitting model...

############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2026-06-28 00:24:35.826265
Success: TRUE

Task duration:
   user  system elapsed 
  0.045   0.035   0.183 

Memory used:
           used  (Mb) gc trigger  (Mb) limit (Mb) max used  (Mb)
Ncells  6233519 333.0   11452545 611.7         NA  9629659 514.3
Vcells 11152177  85.1   21468864 163.8      32768 19634531 149.8

Log messages:
INFO [2026-06-28 00:24:35] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 7
Node: 18
Timestamp: 2026-06-28 00:24:36.774433
Success: TRUE

Task duration:
   user  system elapsed 
  0.082   0.068   0.402 

Memory used:
           used  (Mb) gc trigger  (Mb) limit (Mb) max used  (Mb)
Ncells  6234021 333.0   11452545 611.7         NA  9629659 514.3
Vcells 11154408  85.2   21468864 163.8      32768 19634531 149.8

Log messages:
INFO [2026-06-28 00:24:35] loading futile.logger package

stderr and stdout:


############### LOG 